# AAD Greeks with JAX

Greek talk conflates two independent axes: the pricing engine (Monte Carlo, PDE, Fourier) and the differentiation method (analytic formulas, bump-and-revalue, AAD). AAD is reverse-mode automatic differentiation — {cite:t}`GilesGlasserman2006` brought it to finance for Monte Carlo Greeks — and `jax.grad` is that machinery. The previous chapter derived barrier delta and gamma by hand and bumped vega and theta. Here we differentiate the pricer code instead: one backward sweep, every first-order Greek, at a small constant multiple of one pricing.

The COS pricer of {cite:t}`FangOosterlee2009AM` is smooth in every input, so AAD through it is exact. The Monte Carlo pricer of the same option is not — and `jax.grad` through it fails silently. Both below.

In [1]:
import time
import jax
jax.config.update("jax_enable_x64", True)  # before any jnp array exists; float32 wrecks the recursion
import jax.numpy as jnp
from functools import partial

S0, K, H, T, r, sigma, M = 100.0, 100.0, 120.0, 1.0, 0.05, 0.2, 12
N = 256

Functional port of the FFT barrier pricer from the barrier chapter. Same recursion, three JAX-isms: updates go through `.at[].set()`, the $m_j$ division needs a safe denominator inside `jnp.where` (both branches get differentiated — an unguarded $/j$ at $j=0$ poisons every gradient with NaN), and $N, M$ are static so shapes are known at trace time.

In [2]:
@partial(jax.jit, static_argnums=(6, 7))
def cos_barrier_uoc_jax(S0, K, H, T, r, sigma, N=256, M=12):
    # functional port of the FFT barrier pricer; differentiable in all float args
    x0, h = jnp.log(S0 / K), jnp.log(H / K)
    drift = (r - 0.5 * sigma**2) * T
    half = 10.0 * sigma * jnp.sqrt(T)
    a, b = x0 + drift - half, x0 + drift + half   # _compute_domain(x0, T, 10, sigma, r)
    h = jnp.minimum(h, b)
    bma = b - a
    dt = T / M
    k = jnp.arange(N)
    w = k * jnp.pi / bma
    phi = jnp.exp(1j * w * (r - 0.5 * sigma**2) * dt - 0.5 * sigma**2 * w**2 * dt)

    # chi_k(0,h), psi_k(0,h) -- Eqs. (22)-(23); chi needs no k=0 branch (w=0 gives e^d - e^c)
    chi = (1.0 / (1.0 + w**2)) * (
        jnp.exp(h) * (jnp.cos(w * (h - a)) + w * jnp.sin(w * (h - a)))
        - (jnp.cos(w * (0.0 - a)) + w * jnp.sin(w * (0.0 - a))))
    w_safe = jnp.where(k == 0, 1.0, w)
    psi = jnp.where(k == 0, h - 0.0,
                    (jnp.sin(w_safe * (h - a)) - jnp.sin(w_safe * (0.0 - a))) / w_safe)
    Vk = (2.0 / bma) * (chi - psi)

    # m_j, j = 1-N .. 2N-1, with (x1,x2)=(a,h)
    j = jnp.arange(1 - N, 2 * N)
    j_safe = jnp.where(j == 0, 1, j)
    mj = jnp.where(j == 0,
                   1j * jnp.pi * (h - a) / bma,
                   (jnp.exp(1j * j * jnp.pi * (h - a) / bma) - 1.0) / j_safe)
    off = N - 1
    ms = jnp.zeros(2 * N, dtype=complex)
    ms = ms.at[0].set(mj[off])
    ms = ms.at[1:N].set(mj[off - 1::-1])             # m_{-1} .. m_{1-N}
    ms = ms.at[N + 1:].set(mj[off + N - 1:off:-1])   # m_{N-1} .. m_1
    mc = mj[off + 2 * N - 1:off - 1:-1]              # m_{2N-1} .. m_0
    fft_ms, fft_mc = jnp.fft.fft(ms), jnp.fft.fft(mc)

    zeros = jnp.zeros(N)
    for m in range(M - 1, 0, -1):   # M static -> loop unrolls at trace time
        u = phi * Vk
        u = u.at[0].set(u[0] * 0.5)   # .at[].multiply hits a jax 0.6.2 AD bug under jit+grad
        Msu = jnp.fft.ifft(fft_ms * jnp.fft.fft(jnp.concatenate([u, zeros])))[:N]
        Mcu = jnp.fft.ifft(fft_mc * jnp.fft.fft(jnp.concatenate([zeros, u])))[:N][::-1]
        Vk = jnp.exp(-r * dt) / jnp.pi * jnp.imag(Msu + Mcu)

    F = jnp.real(phi * jnp.exp(1j * w * (x0 - a)))
    F = F.at[0].set(F[0] * 0.5)
    return K * jnp.exp(-r * dt) * jnp.sum(F * Vk)


price = cos_barrier_uoc_jax
print(f"JAX price: {price(S0, K, H, T, r, sigma, N, M):.6f}   (numpy value from the barrier chapter: 1.849354)")

JAX price: 1.849354   (numpy value from the barrier chapter: 1.849354)


Gradients with respect to $(S_0, T, r, \sigma)$ in one call; gamma by differentiating twice. Sign note: the previous chapter's theta is calendar decay $(P(T-\varepsilon)-P(T))/\varepsilon = -\partial P/\partial T$.

In [3]:
dP = jax.grad(price, argnums=(0, 3, 4, 5))
delta, dPdT, rho, vega = dP(S0, K, H, T, r, sigma, N, M)
gamma = jax.grad(jax.grad(price, argnums=0), argnums=0)(S0, K, H, T, r, sigma, N, M)

es = 1e-4
vega_fd = (price(S0, K, H, T, r, sigma + es, N, M) - price(S0, K, H, T, r, sigma - es, N, M)) / (2 * es)
dPdT_fd = (price(S0, K, H, T + es, r, sigma, N, M) - price(S0, K, H, T - es, r, sigma, N, M)) / (2 * es)
rho_fd = (price(S0, K, H, T, r + es, sigma, N, M) - price(S0, K, H, T, r - es, sigma, N, M)) / (2 * es)

print(f"delta  {delta: .6f}   analytic (prev. chapter) -0.010058")
print(f"gamma  {gamma: .6f}   analytic (prev. chapter) -0.007936")
print(f"vega   {vega: .4f}   bump {vega_fd: .4f}   (prev. chapter -15.0120)")
print(f"dP/dT  {dPdT: .4f}   bump {dPdT_fd: .4f}   (prev. chapter theta +1.4120)")
print(f"rho    {rho: .4f}   bump {rho_fd: .4f}")

jax.block_until_ready(dP(S0, K, H, T, r, sigma, N, M))  # warm-up: exclude compile time
t0 = time.time()
for _ in range(50):
    jax.block_until_ready(dP(S0, K, H, T, r, sigma, N, M))
t_grad = (time.time() - t0) / 50 * 1000
t0 = time.time()
for _ in range(50):
    jax.block_until_ready(price(S0, K, H, T, r, sigma, N, M))
t_price = (time.time() - t0) / 50 * 1000
print(f"one gradient sweep (4 Greeks): {t_grad:.2f} ms, one pricing: {t_price:.2f} ms -> {t_grad/t_price:.1f}x")

delta  -0.010058   analytic (prev. chapter) -0.010058
gamma  -0.007936   analytic (prev. chapter) -0.007936
vega   -15.0120   bump -15.0120   (prev. chapter -15.0120)
dP/dT  -1.4111   bump -1.4111   (prev. chapter theta +1.4120)
rho     1.8012   bump  1.8012
one gradient sweep (4 Greeks): 2.48 ms, one pricing: 0.21 ms -> 11.6x


Same option, Monte Carlo pricer, same fixed normals for every evaluation. The price agrees. The AAD delta does not — the knock-out indicator is flat almost everywhere, so its contribution never enters the backward sweep.

In [4]:
Z = jax.random.normal(jax.random.PRNGKey(7), (200_000, M))  # fixed -> common random numbers


def mc_uoc(S0):
    dt = T / M
    logS = jnp.log(S0) + jnp.cumsum((r - 0.5 * sigma**2) * dt + sigma * jnp.sqrt(dt) * Z, axis=1)
    surv = jnp.all(logS < jnp.log(H), axis=1)
    return jnp.exp(-r * T) * jnp.mean(jnp.where(surv, jnp.maximum(jnp.exp(logS[:, -1]) - K, 0.0), 0.0))


def mc_uoc_smooth(S0, width=0.01):
    # sigmoid-smoothed survival: differentiable stand-in for the indicator
    dt = T / M
    logS = jnp.log(S0) + jnp.cumsum((r - 0.5 * sigma**2) * dt + sigma * jnp.sqrt(dt) * Z, axis=1)
    survp = jnp.prod(jax.nn.sigmoid((jnp.log(H) - logS) / width), axis=1)
    return jnp.exp(-r * T) * jnp.mean(survp * jnp.maximum(jnp.exp(logS[:, -1]) - K, 0.0))


print(f"MC price:          {mc_uoc(S0):.4f}   (COS {price(S0, K, H, T, r, sigma, N, M):.4f})")
print(f"AAD through MC:    delta {jax.grad(mc_uoc)(S0): .4f}   (truth {delta: .4f})")
print(f"CRN macro-bump:    delta {(mc_uoc(S0 + 0.5) - mc_uoc(S0 - 0.5)) / 1.0: .4f}")
print(f"smoothed barrier:  delta {jax.grad(mc_uoc_smooth)(S0): .4f}   (sigmoid width 0.01)")

MC price:          1.8498   (COS 1.8494)


AAD through MC:    delta  0.2481   (truth -0.0101)
CRN macro-bump:    delta -0.0037


smoothed barrier:  delta -0.0120   (sigmoid width 0.01)


The pathwise derivative only sees the branch each path already lives on: survivors differentiate like a vanilla, knocked-out paths contribute zero, and the derivative of *which paths survive* — the entire barrier risk, the reason the true delta is small and negative — is a Dirac mass the backward sweep never samples. No error, no warning, wrong sign. The macro bump works because it moves paths across the barrier (0.5% with common random numbers; micro-bumps drown in survival-flip noise), and the smoothed indicator works because it turns the cliff into a differentiable slope — each buys correctness with a bias knob. Production AAD frameworks smooth or condition every discontinuity for exactly this reason.

TODO: AAD through the American max (kinks differentiate fine, jumps don't); `jax.grad` with respect to Heston parameters inside a calibration loop; `lax.scan` + `remat` when M is large enough that unrolling hurts.